[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/quickstart/quickstart_langchain.ipynb)

# DecimalAI + LangChain Quickstart

**Instrument a LangChain agent with 2 lines of code.**

This notebook shows how to add DecimalAI tracing to any LangChain / LangGraph
application. Every LLM call, tool invocation, and chain step is captured automatically.

**Prerequisites:** An OpenAI API key (or any LangChain-supported LLM).

## Step 1 — Install & Configure

In [ ]:
# Install dependencies (the [langchain] extra brings the adapter deps;
# `langchain` provides create_agent, `langchain-openai` the LLM binding)
!pip install -q "decimalai[langchain]" langchain langchain-openai

In [ ]:
import os

# Set your API keys
os.environ["DECIMAL_API_KEY"] = "dai_sk_..."    # ← Get at https://app.decimal.ai/settings
os.environ["OPENAI_API_KEY"] = "sk-..."         # ← Your OpenAI key

# Initialize DecimalAI with LangChain auto-tracing
import decimalai
decimalai.init(langchain=True)

# That's it! All LangChain calls are now traced automatically.

## Step 2 — Build a Simple LangChain Agent

We'll create a tool-calling agent with a couple of tools using langchain 1.x's
`create_agent` (a LangGraph graph under the hood). DecimalAI captures
everything — you don't need to add any extra callbacks or wrappers.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool


# Define tools
@tool
def search_docs(query: str) -> str:
    """Search the knowledge base. Input: search query."""
    return f"Found 3 results for '{query}': [Article 1, Article 2, Article 3]"

@tool
def check_order(order_id: str) -> str:
    """Look up an order status. Input: order ID."""
    return f"Order {order_id}: Shipped on April 25, arriving April 29."

# Create agent
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    llm,
    [search_docs, check_order],
    system_prompt="You are a helpful customer support assistant.",
)

print("✅ LangChain agent ready with 2 tools")

## Step 3 — Run Queries (Traces Are Auto-Captured)

In [ ]:
# Every invocation is automatically traced by DecimalAI
queries = [
    "How do I reset my password?",
    "Where is my order ORD-12345?",
    "What is your return policy?",
]

for q in queries:
    print(f"\n{'='*50}")
    print(f"Q: {q}")
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(f"A: {result['messages'][-1].content}")

print("\n✅ 3 traces auto-captured and sent to DecimalAI!")
print("📊 Open your dashboard: https://app.decimal.ai/traces")

## Step 4 — View in Dashboard

Open **[app.decimal.ai/traces](https://app.decimal.ai/traces)**. For each trace you'll see:

- The full conversation flow (input → LLM → tool calls → output)
- Token usage and latency per LLM call
- Tool call inputs and outputs
- The auto-detected manifest (tools + model)

## Next Steps

- 📖 [Main Quickstart](./quickstart.ipynb) — See the version-aware manifest loop (no LLM key needed)
- 📖 [Evaluations Guide](https://docs.decimal.ai/guides/evaluations) — Score your traces
- 📖 [Training Pipeline](https://docs.decimal.ai/tutorials/training-pipeline) — Build datasets from traces